
# Day 4: Tools, Structured Output and Google Sheets

## Project: Sweet Crumbs Home Bakery AI Agent

### Objective
Build an AI-powered bakery assistant that can:
- Use tools to retrieve product prices.
- Check bakery opening hours.
- Validate and record customer orders.
- Store orders in Google Sheets.
- Convert customer messages into structured data.
- Use execution traces to understand agent behaviour.

### Technologies Used
- Python
- LangChain
- Groq LLM
- Pydantic
- Google Colab
- Google Sheets
- gspread

### Learning Outcomes
Understand how AI agents use tools, how structured output works, how external data can be stored in Google Sheets and how tool execution can be analysed.


In [1]:
!pip install -q langchain langchain-groq gspread

In [2]:
import os
from google.colab import userdata

os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")

from langchain_groq import ChatGroq

llm = ChatGroq(
    model="openai/gpt-oss-20b",
    temperature=0.2,
    max_retries=3
)

In [3]:
from google.colab import auth
auth.authenticate_user()

In [4]:
import gspread
from google.auth import default

creds, _ = default()
gc = gspread.authorize(creds)

sh = gc.create("Sweet Crumbs Orders")
ws = sh.sheet1

ws.append_row(["name", "item", "qty", "date", "eggless", "total"])

print("Sheet ready:", sh.url)

Sheet ready: https://docs.google.com/spreadsheets/d/1wU70HtZIA9Wt5tiI6UBUBDRcZUG-Sjdz_B31D1ZTCIg


In [5]:
print(ws.get_all_values())

[['name', 'item', 'qty', 'date', 'eggless', 'total']]



## 1. Defining Bakery Information

First, we define the bakery's menu and opening hours using Python dictionaries.

These dictionaries act as the source of information for our tools. The agent will use the tools to retrieve this information instead of guessing prices or timings.


In [6]:
MENU = {
    "plum cake": 500,
    "chocolate cake": 650,
    "cupcake": 80
}

HOURS = {
    "monday": "9 am to 8 pm",
    "tuesday": "9 am to 8 pm",
    "wednesday": "9 am to 8 pm",
    "thursday": "9 am to 8 pm",
    "friday": "9 am to 8 pm",
    "saturday": "9 am to 8 pm",
    "sunday": "10 am to 6 pm"
}


## 2. Creating a Structured Order Model

We use Pydantic to define the structure of a customer order.

The Order model contains:
- Customer name
- Menu item
- Quantity
- Order date
- Eggless preference

We also apply restrictions:
- Only three menu items are accepted.
- Quantity must be at least 1.


In [7]:
from typing import Literal
from pydantic import BaseModel, Field

class Order(BaseModel):
    """A cake order taken from a customer's chat message."""

    name: str = Field(description="Customer's first name")

    item: Literal["plum cake", "chocolate cake", "cupcake"] = Field(
        description="Menu item"
    )

    qty: int = Field(description="How many", ge=1)

    date: str = Field(description="Pickup date as the customer said it")

    eggless: bool = Field(description="True if the customer asked for eggless")


## 3. Building Custom Tools

An AI agent can use external functions called tools to perform specific tasks.

We create three tools:

1. get_price – calculates the price of an item.
2. is_open – checks the bakery's opening hours.
3. log_order – validates and saves a customer order to Google Sheets.


In [8]:
from langchain.tools import tool

@tool
def get_price(item: str, qty: int = 1) -> str:
    """Get the total price in rupees for a quantity of one menu item.

    Args:
        item: Menu item, e.g. "plum cake", "chocolate cake" or "cupcake"
        qty: How many the customer wants
    """

    price = MENU.get(item.lower().strip())

    if price is None:
        return f"'{item}' is not on the menu. We have: {', '.join(MENU)}."

    return f"{qty} x {item} = Rs {price * qty}"

In [9]:
print(get_price.invoke({"item": "cupcake", "qty": 3}))
print(get_price.invoke({"item": "plum cake", "qty": 2}))
print(get_price.invoke({"item": "donut", "qty": 1}))

3 x cupcake = Rs 240
2 x plum cake = Rs 1000
'donut' is not on the menu. We have: plum cake, chocolate cake, cupcake.


In [10]:
@tool
def is_open(day: str) -> str:
    """Check the bakery's opening hours for a day of the week, e.g. "sunday"."""

    hours = HOURS.get(day.lower().strip())

    return f"Open {hours} on {day}." if hours else f"'{day}' is not a day of the week."

In [11]:
print(is_open.invoke({"day": "Sunday"}))
print(is_open.invoke({"day": "Monday"}))
print(is_open.invoke({"day": "Holiday"}))

Open 10 am to 6 pm on Sunday.
Open 9 am to 8 pm on Monday.
'Holiday' is not a day of the week.


In [12]:
@tool
def log_order(name: str, item: str, qty: int, date: str, eggless: bool) -> str:
    """Save a confirmed cake order to the Orders sheet.
    Use only when the customer has given their name, item, quantity and pickup date."""

    try:
        order = Order(
            name=name,
            item=item.lower().strip(),
            qty=qty,
            date=date,
            eggless=eggless
        )

    except Exception as e:
        return f"Could not save the order: {e}. Ask the customer to correct it."

    total = MENU[order.item] * order.qty

    ws.append_row([
        order.name,
        order.item,
        order.qty,
        order.date,
        "yes" if order.eggless else "no",
        total
    ])

    return f"Saved: {order.qty} x {order.item} for {order.name} on {order.date}, Rs {total}."

In [13]:
print(log_order.invoke({
    "name": "Ravi",
    "item": "chocolate cake",
    "qty": 2,
    "date": "24 Dec",
    "eggless": True
}))

Saved: 2 x chocolate cake for Ravi on 24 Dec, Rs 1300.


In [14]:
print(ws.get_all_values())

[['name', 'item', 'qty', 'date', 'eggless', 'total'], ['Ravi', 'chocolate cake', '2', '24 Dec', 'yes', '1300']]



## 4. Creating the Bakery AI Agent

We connect the LLM with the three custom tools.

The system prompt defines the agent's role, business information and rules.

The chat function maintains conversation history so that the agent can remember information provided in earlier messages.


In [15]:
from langchain.agents import create_agent

SYSTEM = """ROLE: You are the friendly order assistant for Sweet Crumbs Home Bakery.

FACTS: Orders need 24 hours' notice. All cakes have eggless versions.

RULES:
- Always use get_price for prices and is_open for timings; never guess them.
- Before log_order, make sure you have name, item, quantity and pickup date; ask if missing.
- After saving, read the order back with the total.

FORMAT:
- Under 60 words, friendly, English or Telugu matching the customer."""

agent = create_agent(
    model=llm,
    tools=[get_price, is_open, log_order],
    system_prompt=SYSTEM
)

In [16]:
history = []

def chat(text: str) -> str:
    global history

    try:
        result = agent.invoke({
            "messages": history + [
                {"role": "user", "content": text}
            ]
        })

        history = result["messages"]

        return history[-1].content

    except Exception as e:
        print("Agent error:", e)
        return "Sorry, I'm a little busy. Please try again in a minute."

In [17]:
print(chat("Are you open on Sunday? How much are 2 chocolate cakes?"))

Yes! We’re open Sunday from 10 am to 6 pm. Two chocolate cakes cost ₹1,300. Remember, we need 24 hours’ notice for orders. Could you share your name, pickup date, and quantity so we can confirm?


In [18]:
print(chat("My name is Alex. I want 2 eggless plum cakes."))

Great, Alex! Just let me know the pickup date you’d like for the 2 eggless plum cakes, and I’ll lock it in.


In [19]:
print(chat("I need them on 28 December."))

✅ Order confirmed!  
Alex, 2 eggless plum cakes are ready for pickup on 28 December. Total: ₹1,000.  
We’ll have them waiting for you. Let us know if anything else is needed!


In [20]:
print(ws.get_all_values())


[['name', 'item', 'qty', 'date', 'eggless', 'total'], ['Ravi', 'chocolate cake', '2', '24 Dec', 'yes', '1300'], ['Alex', 'plum cake', '2', '28 December', 'yes', '1000']]


In [21]:
print(chat("I'm John. I want 2 donuts for 29 December."))

Hi John! 👋 Donuts aren’t on our menu—our favorites are plum cake, chocolate cake, and cupcakes. Which one would you like instead? 2 pieces of your chosen item for 29 December will cost ₹… (I’ll fetch the price once you pick). Remember, we need 24 hours’ notice. Let me know!


In [22]:
print(chat("I'm Priya. I want 3 cupcakes."))

Hi Priya! 🎂 Your 3 cupcakes will be ₹240. Could you let me know the pickup date you’d like? Once I have that, I’ll confirm the order.


In [23]:
print(chat("I'm Sam. I want 0 cupcakes for 30 December."))

Hi Sam! 👋 I see you’d like 0 cupcakes on 30 December—just to confirm, is that a mistake or do you need something else? If you’re ordering, we’ll need a positive quantity and 24 hours’ notice. Let me know how I can help!


In [24]:
print(ws.get_all_values())

[['name', 'item', 'qty', 'date', 'eggless', 'total'], ['Ravi', 'chocolate cake', '2', '24 Dec', 'yes', '1300'], ['Alex', 'plum cake', '2', '28 December', 'yes', '1000']]



## 5. Structured Output with Pydantic

Structured output converts a customer's natural-language message into an Order object.

Instead of receiving only a normal text response, we can access individual fields such as name, item, quantity, date and eggless preference.

We test this feature using different customer messages.


In [25]:
extractor = llm.with_structured_output(Order)

In [26]:
order = extractor.invoke(
    "Hi, I'm Ravi. Need 2 eggless plum cakes for 24th December."
)

print(order)

name='Ravi' item='plum cake' qty=2 date='24th December' eggless=True


In [27]:
print(order.name)
print(order.item)
print(order.qty)
print(order.date)
print(order.eggless)

Ravi
plum cake
2
24th December
True


In [28]:
messages = [
    "hey im john need 3 choc cakes for 25 dec eggless",
    "Priya here, can I get one cupcake on 26th December?",
    "hello, want 5 plum cakes for 28 dec, no eggs please",
    "I'm Sam. Need 2 donuts for 30 December",
    "Ravi here, need zero cupcakes for tomorrow"
]

for msg in messages:
    print("\nCustomer message:", msg)

    try:
        order = extractor.invoke(msg)
        print("Extracted order:", order)

    except Exception as e:
        print("Could not extract order:", e)



Customer message: hey im john need 3 choc cakes for 25 dec eggless
Extracted order: name='john' item='chocolate cake' qty=3 date='25 dec' eggless=True

Customer message: Priya here, can I get one cupcake on 26th December?
Extracted order: name='Priya' item='cupcake' qty=1 date='26th December' eggless=False

Customer message: hello, want 5 plum cakes for 28 dec, no eggs please
Could not extract order: Error code: 400 - {'error': {'message': 'Tool choice is required, but model did not call a tool', 'type': 'invalid_request_error', 'code': 'tool_use_failed', 'failed_generation': 'Sure thing! Could you let me know your name so I can place the order for you?'}}

Customer message: I'm Sam. Need 2 donuts for 30 December
Could not extract order: Error code: 400 - {'error': {'message': 'Tool choice is required, but model did not call a tool', 'type': 'invalid_request_error', 'code': 'tool_use_failed', 'failed_generation': 'I’m sorry, but we don’t offer donuts. I can help you place an order for

In [29]:
test_message = "I'm Ravi. I need 2 chocolate cakes for 25 December, eggless."

try:
    order = extractor.invoke(test_message)

    print("Name:", order.name)
    print("Item:", order.item)
    print("Quantity:", order.qty)
    print("Date:", order.date)
    print("Eggless:", order.eggless)

except Exception as e:
    print("Extraction failed:", e)

Name: Ravi
Item: chocolate cake
Quantity: 2
Date: 25 December
Eggless: True



### Structured Output Testing

**Total test messages:** 5

**Successful extractions:** 2

**Failed extractions:** 3

**Observations:**
- The model successfully extracted information from informal and normal customer messages.
- Two messages were converted into structured Order objects.
- Three messages resulted in a Groq tool-use error because the model did not call the required structured-output tool.
- The failed cases included missing customer information, an unavailable menu item and an invalid quantity.
- The API error is different from a direct Pydantic validation error.

**Conclusion:**

Structured output helps convert customer messages into structured Pydantic objects.

However, structured extraction alone is not enough. Additional business validation is necessary before saving an order.



## 6. Reusing Functions as Agent Tools

We convert two previously developed Python functions into LangChain tools.

- calculate_total – calculates the total cost.
- check_availability – checks the 24-hour advance-order requirement.

This demonstrates how ordinary Python functions can become tools for an AI agent.


In [30]:
@tool
def calculate_total(quantity: int, price: float) -> float:
    """Calculate the total price for bakery items.

    Args:
        quantity: Number of items.
        price: Price of one item.
    """
    return quantity * price


@tool
def check_availability(hours_before_order: int) -> str:
    """Check whether an order meets the bakery's 24-hour notice requirement.

    Args:
        hours_before_order: Hours remaining before the requested order time.
    """
    if hours_before_order >= 24:
        return "Order can be accepted."
    return "Order cannot be accepted. Please provide 24 hours' notice."

In [31]:
print(calculate_total.invoke({
    "quantity": 5,
    "price": 80
}))

print(check_availability.invoke({
    "hours_before_order": 30
}))

print(check_availability.invoke({
    "hours_before_order": 12
}))

400.0
Order can be accepted.
Order cannot be accepted. Please provide 24 hours' notice.



## 7. Experiment: Tool Names and Descriptions

We compare a tool with a vague name and description against a tool with a clear, meaningful name and description.

Both tools worked in our experiment.

However, descriptive tool names and docstrings help an AI agent understand the purpose of each tool and when it should be used.


In [32]:
@tool("tool1")
def tool1(item: str) -> str:
    """Does stuff."""
    return f"The price of {item} is Rs 500."

In [33]:
bad_agent = create_agent(
    model=llm,
    tools=[tool1],
    system_prompt="You are a bakery assistant. Use the available tool to answer price questions."
)

In [34]:
result = bad_agent.invoke({
    "messages": [
        {"role": "user", "content": "What is the price of a plum cake?"}
    ]
})

for message in result["messages"]:
    message.pretty_print()

================================ Human Message =================================

What is the price of a plum cake?
================================== Ai Message ==================================
Tool Calls:
  tool1 (fc_1dce02c8-e170-46b8-ac62-7b5f3a416ba8)
 Call ID: fc_1dce02c8-e170-46b8-ac62-7b5f3a416ba8
  Args:
    item: plum cake
================================= Tool Message =================================
Name: tool1

The price of plum cake is Rs 500.
================================== Ai Message ==================================

The price of plum cake is Rs 500.


In [35]:
@tool
def get_plum_cake_price() -> str:
    """Get the current price of one plum cake from Sweet Crumbs Home Bakery."""
    return f"The price of one plum cake is Rs {MENU['plum cake']}."

In [36]:
good_agent = create_agent(
    model=llm,
    tools=[get_plum_cake_price],
    system_prompt="You are a bakery assistant. Use the available tool to answer plum cake price questions."
)

In [37]:
result = good_agent.invoke({
    "messages": [
        {"role": "user", "content": "What is the price of one plum cake?"}
    ]
})

for message in result["messages"]:
    message.pretty_print()



================================ Human Message =================================

What is the price of one plum cake?
================================== Ai Message ==================================
Tool Calls:
  get_plum_cake_price (fc_2a2a243c-3532-4b58-9401-cdb01c6d3614)
 Call ID: fc_2a2a243c-3532-4b58-9401-cdb01c6d3614
  Args:
================================= Tool Message =================================
Name: get_plum_cake_price

The price of one plum cake is Rs 500.
================================== Ai Message ==================================

The price of one plum cake is **Rs 500**.



## 8. Understanding Agent Execution Traces

An execution trace shows the messages exchanged between the user, AI agent and tools.

By analysing the trace, we can identify:
- How many AI messages were generated.
- How many tools were called.
- Which tools were selected.
- What information the tools returned.


In [38]:
tool_calls = 0
ai_messages = 0

for message in history:
    if message.type == "ai":
        ai_messages += 1
        tool_calls += len(message.tool_calls)

print("Total AI messages:", ai_messages)
print("Total tool calls:", tool_calls)

Total AI messages: 12
Total tool calls: 6


In [39]:
for message in history:
    message.pretty_print()

================================ Human Message =================================

Are you open on Sunday? How much are 2 chocolate cakes?
================================== Ai Message ==================================
Tool Calls:
  is_open (fc_7f389d50-1138-46ca-8f12-e35ed05e45f0)
 Call ID: fc_7f389d50-1138-46ca-8f12-e35ed05e45f0
  Args:
    day: sunday
================================= Tool Message =================================
Name: is_open

Open 10 am to 6 pm on sunday.
================================== Ai Message ==================================
Tool Calls:
  get_price (fc_69943dba-8e9f-4452-a4ab-0a55a3260765)
 Call ID: fc_69943dba-8e9f-4452-a4ab-0a55a3260765
  Args:
    item: chocolate cake
    qty: 2
================================= Tool Message =================================
Name: get_price

2 x chocolate cake = Rs 1300
================================== Ai Message ==================================

Yes! We’re open Sunday from 10 am to 6 pm. Two chocolate cakes cost


### Agent Execution Trace Analysis

**Tools used:**
- is_open: 1 call
- get_price: 4 calls
- log_order: 1 call

**Total tool calls:** 6

**Total AI messages:** 12

**Total tool results:** 6

**Observations:**
- The agent called is_open to check opening hours.
- It called get_price to calculate prices and check menu availability.
- It called log_order to save Alex's order.
- It did not save invalid orders.
- It asked follow-up questions when information was missing.

**Conclusion:**

The agent dynamically selected tools based on customer requests instead of relying on fixed responses.



## 9. Creating a Business Facts File

We collect the bakery's menu, prices, opening hours and order-management rules into a separate text file.

This file can serve as a reference for future AI-agent development, business knowledge retrieval and prompt design.


In [40]:

business_facts = """
SWEET CRUMBS HOME BAKERY - BUSINESS FACTS

MENU:
1. Plum cake - Rs 500.
2. Chocolate cake - Rs 650.
3. Cupcake - Rs 80.
4. Eggless versions are available for cakes.

ORDER POLICY:
5. Orders require at least 24 hours' notice.
6. Customer name is collected when placing an order.
7. Product and quantity must be specified.
8. A preferred order date must be provided.
9. The total is calculated using item price multiplied by quantity.

OPENING HOURS:
10. Monday to Saturday: 9 AM to 8 PM.
11. Sunday: 10 AM to 6 PM.

ORDER MANAGEMENT:
12. Orders are recorded in a Google Sheet.
13. The order record includes name, item, quantity,
    date, eggless preference and total.

UNCONFIRMED DETAILS:
14. Delivery availability, delivery locations and
    delivery timings have not been established.
15. Do not promise unverified delivery services.
"""

with open("sweet_crumbs_facts.txt", "w") as file:
    file.write(business_facts)

print("Business facts file created successfully!")


Business facts file created successfully!


In [41]:
from google.colab import files
files.download("sweet_crumbs_facts.txt")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


# Day 4 – Summary and Key Learnings

## What I Built
Developed an AI-powered order assistant for Sweet Crumbs Home Bakery using LangChain, Groq and Google Sheets.

## Key Learnings
1. Created custom tools using LangChain's `@tool` decorator.
2. Used Pydantic to define structured order data.
3. Connected an AI agent with multiple tools.
4. Integrated Google Sheets to store customer orders.
5. Implemented order validation before saving.
6. Used structured output to extract information from customer messages.
7. Analysed agent execution traces to understand tool usage.
8. Learned why clear tool names and descriptions are important.

## Project Outcome
The agent can answer price and opening-hours questions, collect customer order details, calculate totals and save valid orders to Google Sheets.

## Challenges
- Structured output did not work for every test message.
- The agent sometimes required additional information from customers.
- Business validation was necessary before saving orders.

## Conclusion
This workshop helped me understand how an AI agent can use Python functions as tools, work with structured data and interact with external services such as Google Sheets.

**Next Step:** Explore more advanced AI agent capabilities and improve the reliability of the bakery assistant.
